# Fine-tuning OpenAI GPT4.1 mini for MAP Classification

This notebook documents the steps to fine-tune an OpenAI model (in our case GPT 4.1 mini).

## Set up fine-tuning environment 

<div class="alert-warning">
Libraries
</div>

First, import the necessary python packages.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import json
import getpass
import pickle
import json
import os
import time
import tiktoken
import plotnine
from openai import OpenAI
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

<div class="alert-warning">
Set the working directory and login to OpenAI API
</div>

Second, connect to the OpenAI API, using your personal Open AI API key. 

In [ ]:
# Set working directory 
os.chdir('../../../../data')

# If API key is not set in environment variables, prompt the user to enter it
if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass.getpass(prompt='Enter your API key: ')

# Retrieve the API key from environment variables
openai_api_key = os.environ['OPENAI_API_KEY']

# Initialize OpenAI client
client = OpenAI(
  api_key=openai_api_key
)

## Prepare training and validation dataset

Before we can run the fine-tuning, we need to adjust our evaluation dataset such that it can be used for model training. (This part can be skipped. For replication use the resulting evaluation set at the end of this section ("evaluation_set_MAP_sentences_final_v2_2"). This part is the same as in the notebook "Local_Fine_Tuning_final.ipynb")

First, we need to add the confidence score to our evaluation dataset. To do so, we load the evaluation dataset and the outputs of our three best-performing zero shot models. Then we test whether the confidence scores of the three models are broadly in line with our measure of "disclosure quality". If higher disclosure quality is associated with larger confidence scores, then we can use them for training. (For human annotator it is easier to say whether the informational quality of a sentence is "low", "medium", or "high" than coming up with a condfidence score)

In [ ]:
# 0. We first load the evaluation set v4 and the outcomes from the best performing models (GPT-4.1, Llama 70B, Qwen 230B) to create a confidence score for each sentence in the evaluation set
# NOTE: The output results of the best performing models (GPT-4.1, Llama 70B, Qwen 230B) are generated using the final inference prompts (which are slightly different from the prompt engeneering prompts)
# The respective code for generating these outputs can be made available upon request. 

df = pd.read_excel('GLLM/evaluation_set_MAP_sentences_final.xlsx')
df_gpt4 = pd.read_excel('GLLM/Fine_tuning_data/output_ZS_final_gpt-4.1-2025-04-14.xlsx')
df_llama = pd.read_excel('GLLM/Fine_tuning_data/output_ZS_final_Llama-3.3-70B-Instruct.xlsx')
df_qwen = pd.read_excel('GLLM/Fine_tuning_data/output_ZS_final_Qwen3-235B-A22B-Instruct-2507-FP8.xlsx')

# 1. We rename the first column in each dataset to "custom_id" and merge the model outputs 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score' with the evaluation set

df_gpt4 = df_gpt4.rename(columns={df_gpt4.columns[0]: 'custom_id'})
df_llama = df_llama.rename(columns={df_llama.columns[0]: 'custom_id'})
df_qwen = df_qwen.rename(columns={df_qwen.columns[0]: 'custom_id'}) 

# For the human annotated dataset we create the new column 'custom_id' by taking the row index as the value for each row, and place it as the first column in the dataframe
df.insert(0, 'custom_id', df.index)
df['LLM_Confidence_Score'] = float(0) 

df_merged = df.merge(df_gpt4[['custom_id', 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score']], on='custom_id', suffixes=('', '_gpt4'))
df_merged = df_merged.merge(df_llama[['custom_id', 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score']], on='custom_id', suffixes=('', '_llama'))
df_merged = df_merged.merge(df_qwen[['custom_id', 'Explicit_MAP_referral', 'Implicit_MAP_referral', 'LLM_Confidence_Score']], on='custom_id', suffixes=('', '_qwen'))

del df_gpt4, df_llama, df_qwen, df

# 2. We create a new dataframe to calculate the mean confidence score for each model and disclosure quality level of the human annotator
# The rows are the different disclosure quality levels "low", "medium", "high"
# The columns are the different models "gpt4", "llama", "qwen
columns = ['Mean_Confidence_Score_gpt4', 'Mean_Confidence_Score_llama', 'Mean_Confidence_Score_qwen']
rows = ['Low', 'Medium', 'High']
confidence_scores = pd.DataFrame(float(0), index=rows, columns=columns)

for level in rows:
    for model in ['gpt4', 'llama', 'qwen']:
        subset = df_merged[df_merged['Disclosure_quality'] == level]
        mean_confidence = subset[f'LLM_Confidence_Score_{model}'].mean()
        confidence_scores.at[level, f'Mean_Confidence_Score_{model}'] = mean_confidence

print(confidence_scores)

# 3. Next we test whether the confidence scores differ significantly between the different disclosure quality levels for each model using ANOVA
import scipy.stats as stats
low_scores = df_merged[df_merged['Disclosure_quality'] == 'Low'][['LLM_Confidence_Score_gpt4', 'LLM_Confidence_Score_llama', 'LLM_Confidence_Score_qwen']]
medium_scores = df_merged[df_merged['Disclosure_quality'] == 'Medium'][['LLM_Confidence_Score_gpt4', 'LLM_Confidence_Score_llama', 'LLM_Confidence_Score_qwen']]
high_scores = df_merged[df_merged['Disclosure_quality'] == 'High'][['LLM_Confidence_Score_gpt4', 'LLM_Confidence_Score_llama', 'LLM_Confidence_Score_qwen']]

for model in ['gpt4', 'llama', 'qwen']:
    # Perform ANOVA test low vs. medium
    f_statistic, p_value = stats.f_oneway(low_scores[f'LLM_Confidence_Score_{model}'], medium_scores[f'LLM_Confidence_Score_{model}'], high_scores[f'LLM_Confidence_Score_{model}'])
    print(f'ANOVA results "low" vs "medium" for {model}: F-statistic = {f_statistic}, p-value = {p_value}')
    #Perform ANOVA test medium vs. high
    f_statistic, p_value = stats.f_oneway(medium_scores[f'LLM_Confidence_Score_{model}'], high_scores[f'LLM_Confidence_Score_{model}'])
    print(f'ANOVA results "medium" vs "high" for {model}: F-statistic = {f_statistic}, p-value = {p_value}')

Next, we compute the confidence scores for model fine-tuning. (Note: It yields evaluation set v5, which is based on v4 but also includes the Confidence Score. The score is based on the output of GPT 4.1, Llama 70B, and Qwen 230B where the score is the average from at least two models that have the same output for explicit and implicit as the human annotator and manually adapted where no model has the same output)

In [ ]:
# 1. Create a new column for each model indicating whether the model's output matches the human annotation
df_merged['gpt4_match'] = ((df_merged['Explicit_MAP_referral'] == df_merged['Explicit_MAP_referral_gpt4']) & (df_merged['Implicit_MAP_referral'] == df_merged['Implicit_MAP_referral_gpt4'])).astype(int)
df_merged['llama_match'] = ((df_merged['Explicit_MAP_referral'] == df_merged['Explicit_MAP_referral_llama']) & (df_merged['Implicit_MAP_referral'] == df_merged['Implicit_MAP_referral_llama'])).astype(int)
df_merged['qwen_match'] = ((df_merged['Explicit_MAP_referral'] == df_merged['Explicit_MAP_referral_qwen']) & (df_merged['Implicit_MAP_referral'] == df_merged['Implicit_MAP_referral_qwen'])).astype(int)

# 2. Create a column that indicates the the number of models that do match the human annotation
df_merged['num_match'] = df_merged['gpt4_match'] + df_merged['llama_match'] + df_merged['qwen_match']


# 3. Calculate the average confidence score of the models confidence score for each sentence where at least two model's output matches the human annotation. 
# The output is stored in the column 'LLM_Confidence_Score' and should be in steps of 5 (e.g. 0, 5, 10, ..., 100). If less than two models match the human annotation, the confidence score is set to 0.
def calculate_average_confidence(row):
    confidence_scores = []
    if row['gpt4_match'] == 1:
        confidence_scores.append(row['LLM_Confidence_Score_gpt4'])
    if row['llama_match'] == 1:
        confidence_scores.append(row['LLM_Confidence_Score_llama'])
    if row['qwen_match'] == 1:
        confidence_scores.append(row['LLM_Confidence_Score_qwen'])
    
    if len(confidence_scores) >= 2:
        average_score = np.mean(confidence_scores)
        # Round to nearest multiple of 5
        rounded_score = round(average_score / 5) * 5
        return int(rounded_score)
    else:
        return int(0)

df_merged['LLM_Confidence_Score'] = df_merged.apply(calculate_average_confidence, axis=1)

# 4. Create a column that indicates whether manual adjustment is necessary (num_match < 2, or confidence score < 50 but Implicit 'Yes' or < 70 and Explicit 'Yes')
def needs_manual_adjustment(row):
    if row['num_match'] < 2:
        return 'Yes'
    if row['LLM_Confidence_Score'] < 50 and row['Implicit_MAP_referral'] == 'Yes':
        return 'Yes'
    if row['LLM_Confidence_Score'] < 70 and row['Explicit_MAP_referral'] == 'Yes':
        return 'Yes'
    if row['Explicit_MAP_referral'] == 'No' and row['Implicit_MAP_referral'] == 'No' and row['LLM_Confidence_Score'] >= 50:
        return 'Yes'
    return 'No'

df_merged['manual_adjustment'] = df_merged.apply(needs_manual_adjustment, axis=1)

# 5. Save the updated evaluation set with confidence scores and manual adjustment flags
df_merged.to_excel('GLLM/Fine_tuning_data/evaluation_set_MAP_sentences_final_v2.xlsx', index=False)


## Generate training and validation dataset

After manually adjusting the confidence scores of the flagged obersvations, we split the evluation set into a trainings and validation dataset and save it as jsonl format. Note that non-flagged scores were also subject to screening and minor changes could have been done as well. The final evaluation set with manual adjustments is stored in the file "evaluation_set_MAP_sentences_final_v2_2.xlsx".

In [ ]:
# 1. Load the evaluation set v2_2 which includes the confidence scores (with manual adjustments)
df = pd.read_excel('GLLM/Fine_tuning_data/evaluation_set_MAP_sentences_final_v2_2.xlsx')

# 2. Create a new column merging the MAP dimensions from columns "MAP_dimension_1", "MAP_dimension_2", "MAP_dimension_3", and "MAP_dimension_4" into one string separated by commas
def merge_map_dimensions(row):
    return ", ".join([row[f"MAP_dimension_{i}"].strip() for i in range(1, 5) if pd.notna(row[f"MAP_dimension_{i}"])])
df['MAP_dimensions_merged'] = df.apply(merge_map_dimensions, axis=1)

# 3. Save the merged dimensions column as new dataset 
df.to_excel('GLLM/Fine_tuning_data/training_and_validation_data_MAP_sentences.xlsx', index=False)

# 4. Split the data into training and validation set and save as separate dataframes (80% train, 20% validation)
# Set random seed for reproducibility
train_df = df.sample(frac=0.8, random_state=42)

train_df.to_excel('GLLM/Fine_tuning_data/training_data_MAP_sentences.xlsx', index=False)

validation_df = df.drop(train_df.index)

validation_df.to_excel('GLLM/Fine_tuning_data/validation_data_MAP_sentences.xlsx', index=False)

# 5. Define prompt and completion templates

prompt_template = """Text:
{text}
####
"""

completion_template = """
{map_classification}
<|end|>
""".strip()

# 6. create a jsonl file for training and validation 
file_name = "GLLM/Fine_tuning_data/training_data_MAP_sentences_OpenAI.jsonl"

train_input = []


training_file = open(file_name, "w", encoding="utf-8")

for index, row in train_df.iterrows():
    merged_dimension = row['MAP_dimensions_merged']
    map_classification = f"{{\n  \"Explicit_MAP_referral\": \"{row['Explicit_MAP_referral']}\",\n  \"Implicit_MAP_referral\": \"{row['Implicit_MAP_referral']}\",\n  \"Dimension\": \"{merged_dimension}\",\n  \"Confidence_Score\": {row['LLM_Confidence_Score']}\n}}"
    entry = {
        "messages": [
            {"role": "user", "content": prompt_template.format(text=row['Sentence'])},
            {"role": "assistant", "content": completion_template.format(map_classification=map_classification)}
        ]
    }
    training_file.write(json.dumps(entry) + "\n")
    training_file.flush()

training_file.close()


validation_file_name = "GLLM/Fine_tuning_data/validation_data_MAP_sentences_OpenAI.jsonl"

validation_file = open(validation_file_name, "w", encoding="utf-8")

for index, row in validation_df.iterrows():
    merged_dimension = row['MAP_dimensions_merged']
    map_classification = f"{{\n  \"Explicit_MAP_referral\": \"{row['Explicit_MAP_referral']}\",\n  \"Implicit_MAP_referral\": \"{row['Implicit_MAP_referral']}\",\n  \"Dimension\": \"{merged_dimension}\",\n  \"Confidence_Score\": {row['LLM_Confidence_Score']}\n}}"
    entry = {
        "messages": [
            {"role": "user", "content": prompt_template.format(text=row['Sentence'])},
            {"role": "assistant", "content": completion_template.format(map_classification=map_classification)}
        ]
    }
    validation_file.write(json.dumps(entry) + "\n")
    validation_file.flush()

validation_file.close()

We can have a quick look at the training and validation files for fine-tuning.

In [ ]:
training_jsonl_file = "GLLM/Fine_tuning_data/training_data_MAP_sentences_OpenAI.jsonl"
validation_jsonl_file = "GLLM/Fine_tuning_data/validation_data_MAP_sentences_OpenAI.jsonl"

# Load training dataset
with open(training_jsonl_file) as f:
    training_dataset = [json.loads(line) for line in f]

# Load validation dataset
with open(validation_jsonl_file) as f:
    validation_dataset = [json.loads(line) for line in f]

# Check the number of examples and print the first item in the dataset
print("Num examples training:", len(training_dataset))
print("Num examples validation:", len(validation_dataset))
print("First example training:")
for message in training_dataset[0]["messages"]:
    print(message)

print("First example validation:")
for message in validation_dataset[0]["messages"]:
    print(message)

del training_dataset, validation_dataset

## Start fine-tuning 

Now we select the model for fine-tuning and upload the training and validation files to the OpenAI API platform.

In [ ]:

# Specify the model to use for training. In this case, we are using the "gpt-4.1-mini-2025-04-14" model for fine-tuning. 
# This model is a smaller variant of GPT-4.1, which is suitable for tasks that require less computational resources while still providing strong performance.
openai_model = "gpt-4.1-mini-2025-04-14"  

# Upload the training and validation files to OpenAI
ft_file = client.files.create(
        file=open(training_jsonl_file, "rb"),
        purpose="fine-tune"
    )

print("Uploaded training file ID:", ft_file.id)

val_file = client.files.create(
        file = open(validation_jsonl_file, "rb"),
        purpose="fine-tune"
)

print("Uploaded validation file ID:", val_file.id)


Select the hyperparameters and create the fine-tuning job.

In [ ]:
# Baseline Hyperparameters
n_epochs = 5 # Number of epochs, options in the paper [5, 10]
batch_size = 5 # Batch size 
lr_multiplier = 2   # Learning rate multiplier, options in the paper [2 , 0.1]

train_file_id = ft_file.id
val_file_id = val_file.id

fine_tune_job = client.fine_tuning.jobs.create(
    training_file = train_file_id,
    validation_file = val_file_id,
    model = openai_model,
    hyperparameters = {
        "n_epochs": n_epochs,
        "batch_size": batch_size,
        "learning_rate_multiplier": lr_multiplier
    },
    suffix = f"openai-ft-n_epochs-{n_epochs}"
)

print("Fine-tune job created:", fine_tune_job.id , "\nusing the model:", fine_tune_job.model, "\nwith fine-tuning parameters:", fine_tune_job.hyperparameters)

We can track the progress of the fine-tuning job.

In [ ]:
# track progress of fine-tuning job

job_id = fine_tune_job.id
status = "pending" 
while status not in ["succeeded", "failed"]:
    job_info = client.fine_tuning.jobs.retrieve(job_id)
    status = job_info.status
    if status == "succeeded":
        print("Fine-tune job completed successfully.")
        print("Fine-tuned model ID:", job_info.fine_tuned_model)
        break
    print(f"Fine-tune job status: {status}")
    time.sleep(60)  # wait for 1 minute before checking the status again

fine_tuned_model_id = job_info.fine_tuned_model


## Run inference with fine-tuned model on validation dataset

Now use the fine-tuned model on the validation dataset.

First, we load the validation set.

In [ ]:
# load validation dataset
val_file = "GLLM/Fine_tuning_data/validation_data_MAP_sentences.xlsx" 
val_df = pd.read_excel(val_file)

#rename the first column to "Sentence_ID"
val_df.rename(columns={val_df.columns[0]: "Sentence_ID"}, inplace=True)

val_df['custom_id'] = f"eval_sentence_" + val_df['Sentence_ID'].astype(str)

Second, we define a function to create the task batches, a function that runs the batch processing of the validation set, and a fuction to retrieve the results of the batch jobs and process them accordingly..

In [ ]:
# Create batch tasks for evaluation
def create_MAP_batch(data, model="gpt-4.1-nano-2025-04-14"):
    tasks = []
    sentence_column = "Sentence"

    for index, row in data.iterrows():
        sentence = row[sentence_column]
        custom_id = row['custom_id']  # Access the custom_id

        if not sentence or pd.isna(sentence):  # Skip empty or NaN sentences
            continue

        task = {
            "custom_id": f"{custom_id}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model,
                "temperature": 0.001,
                "stop": ["<|end|>"],
                "messages": [
                    {
                        "role": "user",
                        "content": f'{prompt_template.format(text=sentence)}'
                    }
                ],
            }
        }
        tasks.append(task)
    
    return tasks

# Batch processing function
def batch_processing(batch, batch_file_name="evaluation_batch.jsonl"):
    batch_file_name = batch_file_name
    batch_file_id = None
    batch_job_id = None

    # 1. Save each batch to a separate .jsonl file

    with open(batch_file_name, 'w') as file:
        for task in batch:
            file.write(json.dumps(task) + '\n')

    
    # 2. Upload batch files to the API
    batch_file = client.files.create(
        file=open(batch_file_name, "rb"),
        purpose="batch"
    )
    print(f"Batch file uploaded: {batch_file.id}")
    batch_file_id = batch_file.id
    
    # 3. Start batch jobs
    batch_job = client.batches.create(
        input_file_id=batch_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )
    print(f"Batch job started: {batch_job.id}")
    batch_job_id = batch_job.id

    return batch_job_id  # Return the batch job ID

# Function to check batch job status, to wait for completion
def wait_for_batch_to_complete(batch_job_id):
    while True:
        completed_batches = 0
        failed_batches = 0
        total_batches = 1 # alternative: len(batch_job_ids)

        # Track failed batch job IDs
        failed_batch_ids = []

        batch_job = client.batches.retrieve(batch_job_id)
        status = batch_job.status
        print(f"Batch Job {batch_job_id} Status: {status}")

        if status == "completed":
            completed_batches += 1
            print(f"Batch Job {batch_job_id} completed successfully.")
            start_time = batch_job.in_progress_at
            end_time = batch_job.completed_at
            if start_time and end_time:
                duration = (end_time - start_time) / 60 
                print(f"Batch Job {batch_job_id} duration: {duration} minutes")
            return  # Exit the function upon completion
        if status == "in_progress":
            in_progress_at = batch_job.in_progress_at
            if in_progress_at:
                readable_time = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(in_progress_at))
                print(f"Batch Job {batch_job_id} started at: {readable_time}")

        elif status in ["failed", "cancelled", "expired"]:
            failed_batches += 1
            failed_batch_ids.append(batch_job_id)  # Store failed job ID
            print(f"Batch Job {batch_job_id} failed with status: {status}")
            return  # Exit the function upon failure

        # Wait before checking again
        print(f"Waiting 60 seconds before checking again...")
        time.sleep(60)

# Retrieve and parse batch results
def retrieve_batch_results(batch_job_id, model="gpt-4.1-nano-2025-04-14"):
    results = []

    # Normalize values
    def normalize(value):
        if isinstance(value, str):
            value = value.strip()
            if value.lower() in ["n/a", "", "na", "none", "nan"]:
                return None
        return value

    # Get batch job status
    batch_job_status = client.batches.retrieve(batch_job_id)
    result_file_id = batch_job_status.output_file_id

    # ! think about error handeling !
    if not result_file_id:
        print(f"No output file found for batch job {batch_job_id}")

    # Download the batch results file
    result_content = client.files.content(result_file_id).content
    result_file_name = f"GLLM/OpenAI_prompting_results/evaluation_batch_results_{model}.jsonl"

    with open(result_file_name, 'wb') as file:
        file.write(result_content)

    print(f"Batch results downloaded: {result_file_name}")

    # Read and parse the results
    with open(result_file_name, 'r') as file:
        for line in file:
            try:
                response = json.loads(line.strip())
                custom_id = response['custom_id']  
                    
                # Extract rating from response
                body = response.get("response", {}).get("body", {})
                choices = body.get("choices", [])
                usage = body.get("usage", {})
                if choices:
                    content = json.loads(choices[0]["message"]["content"])
                    Explicit = normalize(content.get('Explicit_MAP_referral', None))
                    Implicit = normalize(content.get('Implicit_MAP_referral', None))
                    Dimension = normalize(content.get('Dimension', None))
                    probability = int(content.get('Confidence_Score', None))

                else:
                    print(f"No choices found in response for {custom_id}.")
                    probability = None
                    Explicit = None
                    Implicit = None
                    Dimension = None

                if usage:
                    prompt_tokens = usage.get("prompt_tokens", 0)
                    completion_tokens = usage.get("completion_tokens", 0)
                    total_tokens = usage.get("total_tokens", 0)

                # Append results
                results.append({"custom_id": custom_id, "LLM_Explicit_MAP_referral": Explicit, "LLM_Implicit_MAP_referral": Implicit, "LLM_Dimension": Dimension, "LLM_Confidence_Score": probability, "Input_tokens": prompt_tokens, "Output_tokens": completion_tokens, "Total_tokens": total_tokens})

            except Exception as e:
                response = json.loads(line.strip())
                custom_id = response['custom_id']
                print(f"Error parsing result for {custom_id}: {e}")

    return pd.DataFrame(results)

# Retrieve train and validation data from the training events
def retrieve_training_data(fine_tune_job_id, hyperparameters):

    events = client.fine_tuning.jobs.list_events(fine_tune_job_id, limit=10000)

    # Prepare lists to store metrics
    train_steps = []
    training_losses = []
    full_valid_steps = []
    full_valid_losses = []
    full_valid_accuracies = []

    # Iterate through events and extract metrics
    for event in events.data:
        if event.type == "metrics":
            metrics = event.data
            train_steps.append(metrics.get("step"))
            training_losses.append(metrics.get("train_loss"))
            # some times there is also the variables for valid and full valid
            if "full_valid_loss" in metrics:
                full_valid_steps.append(metrics.get("step"))
                full_valid_losses.append(metrics.get("full_valid_loss"))
                full_valid_accuracies.append(metrics.get("full_valid_mean_token_accuracy"))

    # Create DataFrames for easier handling
    train_metrics_df = pd.DataFrame({"Steps": train_steps, 
                                    "Loss": training_losses, 
                                    "Type": "Train",
                                    "Hyperparameters": f"({hyperparameters['epochs']}, {hyperparameters['batch_size']}, {hyperparameters['learning_rate_multiplier']})"})
    full_valid_metrics_df = pd.DataFrame({"Steps": full_valid_steps, 
                                        "Loss": full_valid_losses, 
                                        "Type": "Validation",
                                        "Hyperparameters": f"({hyperparameters['epochs']}, {hyperparameters['batch_size']}, {hyperparameters['learning_rate_multiplier']})"})

    # Sort DataFrames by step
    train_metrics_df.sort_values("Steps", inplace=True)
    full_valid_metrics_df.sort_values("Steps", inplace=True)

    # Combine the training and full validation metrics into one DataFrame
    df_combined = pd.concat([train_metrics_df, full_valid_metrics_df], ignore_index=True)
    return df_combined

Third, we create the task batches and start the batch processing.

In [ ]:
# Create tasks for the fine-tuned model
Map_tasks = create_MAP_batch(val_df, model = fine_tuned_model_id)

# Process the batch and get the job ID
batch_job_id = batch_processing(Map_tasks, batch_file_name=f"GLLM/OpenAI_batch_files/evaluation_batch_MAP_{fine_tuned_model_id.replace(':', '_')}.jsonl") 

Check the status of the batch job.

In [ ]:
wait_for_batch_to_complete(batch_job_id)

Fourth, when job is finished, retrieve the results from the API database and join it to the main dataframe.

In [ ]:
# Retrieve the results of the batch job
df_results = retrieve_batch_results(batch_job_id, model=fine_tuned_model_id.replace(':', '_'))

#remove all observations where LLM_Explicit_MAP_referral and/or LLM_Implicit_MAP_referral is not None, Yes, or No
beginning_count = len(df_results)
print(f"Initial number of results: {beginning_count}")

df_results = df_results[df_results['LLM_Explicit_MAP_referral'].isin(['Yes', 'No', None])]
df_results = df_results[df_results['LLM_Implicit_MAP_referral'].isin(['Yes', 'No', None])]
df_results = df_results.dropna(subset=['LLM_Explicit_MAP_referral', 'LLM_Implicit_MAP_referral'])
ending_count = len(df_results)
print(f"Number of results after filtering: {ending_count}")

if beginning_count - ending_count > 0:
    print(f"Removed {beginning_count - ending_count} results due to invalid values in LLM_Explicit_MAP_referral or LLM_Implicit_MAP_referral.")

# Merge the results with the validation dataframe on 'custom_id'
df_final = val_df.merge(df_results, on='custom_id', how='left')

# Save the final results to an Excel file
df_final.to_excel(f"GLLM/OpenAI_prompting_results/output_FT_val_{fine_tuned_model_id.replace(':', '_')}.xlsx", index=False)

# retrieve training data for the fine-tuned model
hyperparameters = {
    "epochs": n_epochs,
    "batch_size": batch_size,
    "learning_rate_multiplier": lr_multiplier
}
training_data_df = retrieve_training_data(fine_tune_job_id=fine_tune_job.id, hyperparameters=hyperparameters)

# Save the training data to an Excel file
training_data_df.to_excel(f"GLLM/OpenAI_prompting_results/training_data_FT_{fine_tuned_model_id.replace(':', '_')}.xlsx", index=False)

Fifth, to better compare the performance of the fine-tuned model to the zero-shot models, we run the performance evaluation of the latter on the validation subset.

In [ ]:
def evaluate_llm(dataset, model_id="gpt-4.1-mini-2025-04-14", system_idx=1, user_idx=2,
                          truth_exp_col="Explicit_MAP_referral", pred_exp_col="LLM_Explicit_MAP_referral",
                          truth_imp_col="Implicit_MAP_referral", pred_imp_col="LLM_Implicit_MAP_referral",
                          truth_dimension_col="MAP_dimension_1", pred_dimension_col="LLM_Dimension"):

    # Create a copy of the dataset to avoid modifying the original
    df = dataset.copy()
    
    # Drop rows with missing LLM predictions
    df_exp = df.dropna(subset=[truth_exp_col, pred_exp_col]).copy()
    df_imp = df.dropna(subset=[truth_imp_col, pred_imp_col]).copy()

    print(f"Dropped {len(df)-len(df_exp)} explicit / {len(df)-len(df_imp)} implicit sentences")
    
    #Evaluate Explicit MAP Referral
    print("=== Explicit MAP Referral Evaluation ===")
    if not df_exp.empty:
        exp_accuracy = accuracy_score(df_exp[truth_exp_col], df_exp[pred_exp_col])
        exp_f1_yes = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_precision_yes = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_recall_yes = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_f1_no = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_precision_no = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_recall_no = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Explicit):")
        print(classification_report(df_exp[truth_exp_col], df_exp[pred_exp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Explicit MAP evaluation.")
    
    #Evaluate Implicit MAP Referral
    print("\n=== Implicit MAP Referral Evaluation ===")
    if not df_imp.empty:
        imp_accuracy = accuracy_score(df_imp[truth_imp_col], df_imp[pred_imp_col])
        imp_precision_yes = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_recall_yes = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_f1_yes = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_precision_no = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_recall_no = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_f1_no = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Implicit):")
        print(classification_report(df_imp[truth_imp_col], df_imp[pred_imp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Implicit MAP evaluation.")

    #Evaluate MAP Dimension
    print("\n=== MAP Dimension Evaluation ===")
    filtered_dimension = df[
        (df["LLM_Explicit_MAP_referral"] == "No") &
        (df["LLM_Implicit_MAP_referral"] == "No") &
        (~df["LLM_Dimension"].isna())
    ]
    dim_percentage = len(filtered_dimension)/len(df)*100
    print(f"Number of rows where LLM says 'No' to both Explicit and Implicit MAP referral but MAP Dimension is not None: {dim_percentage:.0f}%")

    # Create a list of all dimension columns
    dimension_cols = [col for col in df.columns if col.startswith("MAP_dimension")]
    
    # Check if the micro / macro F1 score for the dimension column 
    label_space = ["Budgeting / Planning", "Cost", "Financing / Investment", "Operations", "Performance / Internal Reporting", "Risk / Internal Control", "Strategy", "Pricing & Revenue Management"]
    
    def encode_labels(text, label_space):
        labels = [l.strip() for l in text.split(",")]
        return [1 if label in labels else 0 for label in label_space]
    
    # join the true cols without empty cells to one column and encode the true and pred dimension cols

    df["dimension_true"] = df[dimension_cols].apply(lambda row: ", ".join(row.dropna().astype(str)), axis=1)
    df["dimension_true_encoded"] = df["dimension_true"].apply(lambda text: encode_labels(text, label_space))
    df["dimension_pred_encoded"] = df[pred_dimension_col].apply(lambda text: encode_labels(text, label_space) if isinstance(text, str) else [0]*len(label_space))

    dimension_micro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="micro", zero_division=0)
    dimension_macro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="macro", zero_division=0)
    dimension_accuracy = accuracy_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist())
    #full report as table
    dimension_full_report = classification_report(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), target_names=label_space, zero_division=0, digits=3)
    
    # In addition, we check if at least one true dimension is included in the predicted dimensions (which may be a comma-separated string)
    def check_dimension_match(row):
        #true_dim = row[truth_dimension_col]
        true_dims = [row[col] for col in dimension_cols if pd.notnull(row[col])]
        pred_raw = row[pred_dimension_col]
        if all(pd.isnull(true_dims)) and pd.isnull(pred_raw):
            return True  # Both are NaN = match
        elif pd.isnull(pred_raw):
            return False  # No prediction = no match
        else:
            pred_dims = [dim.strip() for dim in pred_raw.split(",")]
            return any(true_dim in pred_dims for true_dim in true_dims)

    if not df.empty:
        print("\nFull per-label Dimension classification report:\n")
        print(dimension_full_report)
        print(f"MAP Dimension Accuracy:{dimension_accuracy:.3f}")
        df["dimension_match"] = df.apply(check_dimension_match, axis=1)
        dimension_accuracy_alternative = df["dimension_match"].mean()
        print(f"MAP Dimension Accuracy (both N/A = match, at least one match): {dimension_accuracy_alternative:.3f}")
    else:
        print("No valid rows for MAP Dimension evaluation.")

    # Return the evaluation results as a dictionary
    return {
        "model_id": model_id,
        "system_idx": system_idx,
        "user_idx": user_idx,
        "dropped_explicit": len(df) - len(df_exp),
        "dropped_implicit": len(df) - len(df_imp),
        "explicit_accuracy": exp_accuracy,
        "explicit_precision_yes": exp_precision_yes,
        "explicit_recall_yes": exp_recall_yes,
        "explicit_f1_yes": exp_f1_yes,
        "explicit_precision_no": exp_precision_no,
        "explicit_recall_no": exp_recall_no,
        "explicit_f1_no": exp_f1_no,
        "implicit_accuracy": imp_accuracy,
        "implicit_precision_yes": imp_precision_yes,
        "implicit_recall_yes": imp_recall_yes,
        "implicit_f1_yes": imp_f1_yes,
        "implicit_precision_no": imp_precision_no,
        "implicit_recall_no": imp_recall_no,
        "implicit_f1_no": imp_f1_no,
        "dimension_percentage_false": dim_percentage,
        "dimension_micro_f1": dimension_micro_f1,
        "dimension_macro_f1": dimension_macro_f1,
        "dimension_accuracy": dimension_accuracy,
        "dimension_accuracy_alternative": dimension_accuracy_alternative,
        "dimension_full_report": dimension_full_report
    }

# load the validation data set
val_data_path = "GLLM/Fine_tuning_data/validation_data_MAP_sentences.xlsx"
val_df = pd.read_excel(val_data_path)

#rename the first column to "Sentence_ID"
val_df.rename(columns={val_df.columns[0]: "Sentence_ID"}, inplace=True)

# list all output file of the fine-tuned models (starting with "output_FT_val_")
fine_tuned_output_files = [f"OpenAI_prompting_results/{f.split('.xlsx')[0]}" for f in os.listdir("GLLM/OpenAI_prompting_results") if f.startswith("output_FT_val_") and f.endswith(".xlsx")]

# load the output from previous best performing LLM inferences
file_names = ["Local_prompting_results/output_ZS_sys1_user2_Qwen3-4B-Instruct-2507", "Local_prompting_results/output_ZS_sys1_user2_Llama-3.3-70B-Instruct", "Local_prompting_results/output_ZS_sys1_user2_Qwen3-235B-A22B-Instruct-2507-FP8",
              "OpenAI_prompting_results/output_ZS_sys1_user2_gpt-4.1-mini-2025-04-14", "OpenAI_prompting_results/output_ZS_sys1_user2_gpt-4.1-2025-04-14"] + fine_tuned_output_files

evaluation_results = [] 

for file_name in file_names:
    llm_output_path = f"GLLM/{file_name}.xlsx"
    if file_name.startswith("OpenAI_prompting_results/output_FT_val_ft_"):
        model_id = file_name.split("output_FT_val_ft_")[-1]
    else:
        model_id = file_name.split("_")[-1]
    print(f"Evaluating LLM output from: {llm_output_path}")
    llm_output_df = pd.read_excel(llm_output_path)

    # rename the first column to "Sentence_ID"
    llm_output_df.rename(columns={llm_output_df.columns[0]: "Sentence_ID"}, inplace=True)
    # merge the previous output (column: LLM_Explicit_MAP_referral, LLM_Implicit_MAP_referral, LLM_Dimension, Confidence_Score) with the evaluation data
    merged_df = pd.merge(val_df, llm_output_df[["Sentence_ID", "LLM_Explicit_MAP_referral", "LLM_Implicit_MAP_referral", "LLM_Dimension", "LLM_Confidence_Score"]], on="Sentence_ID", how="left")
    if file_name.startswith("OpenAI_prompting_results/output_FT_val_ft_"):
        evaluation_result = evaluate_llm(merged_df, model_id=model_id, system_idx=0, user_idx=3)
    else:
        evaluation_result = evaluate_llm(merged_df, model_id=model_id, system_idx=1, user_idx=2)
    evaluation_results.append(evaluation_result)

pd.DataFrame(evaluation_results).to_excel("GLLM/validation_summary_total.xlsx", index=False)

Lastly, we create one plot that shows the training and validation losses for the most interesting models, in order to compare them.

In [ ]:
# Create a custom color palette for the plotnine plots
custom_color_palette  = ['#377eb8', '#ff7f00', '#4daf4a','#a65628', '#984ea3', '#e41a1c', '#dede00']

# Create plot folder if it doesn't exist
os.makedirs("Plots", exist_ok=True)

# NOTE: Please check the job IDs and hyperparameters to ensure they match your actual fine-tuning jobs and configurations. 
# Adjust the job_ids and hyperparameters dictionaries as needed based on your specific fine-tuning runs.
# Example: "ft_gpt-4.1-mini-2025-04-14_personal_openai-ft-n-epochs-5_CYr7J5l9"
model_ids = ["ft_gpt-4.1_Example1", "ft_gpt-4.1_Example2", "ft_gpt-4.1_Example3"]  # Replace with your actual fine-tuning model IDs

# Set up the plotnine theme
plotnine.options.figure_size = (12, 6)

# Create an empty DataFrame to hold all the data for plotting
df_all = pd.DataFrame()

trail_number = 1

# Iterate through each job ID and extract metrics
for model_id in model_ids:
    # Retrieve the previous saved fine-tuning job details
    fine_tune_df = pd.read_excel(f"GLLM/OpenAI_prompting_results/training_data_FT_{model_id.replace(':', '_')}.xlsx")

    #add a column for the trial number
    fine_tune_df["Trial"] = f"Trial {trail_number}"

    trail_number += 1

    # Add the current job's data to the overall DataFrame for plotting
    df_all = pd.concat([df_all, fine_tune_df], ignore_index=True)

# just use datapoints for type "Train" in steps of 25 for better visibility of the plot and use all datapoints for the validation loss
df_all = df_all[(df_all["Type"] == "Validation") | ((df_all["Type"] == "Train") & (df_all["Steps"] % 25 == 0))]

# just use loss values below 2 to focus on the relevant part of the plot
df_all = df_all[df_all["Loss"] < 2]

# Plot the training and full validation loss over steps for all jobs using plotnine
# Create the plot using plotnine
# make sure to order the legend by trial number and put the legend on the right side of the plot (in the order of the trials_to_plot list)
p = (plotnine.ggplot(df_all, plotnine.aes(x="Steps", y="Loss", color="Trial", linetype="Type"))
     + plotnine.geom_line(size=1.5)
     + plotnine.scale_color_manual(values=custom_color_palette)
     + plotnine.labs(x="Training Steps",  y="Loss")
     + plotnine.theme_minimal()
     + plotnine.theme(
            legend_position="right",
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_x=plotnine.element_text(size=14, weight='bold'),  
            axis_text_y=plotnine.element_text(size=14, weight='bold'))
     + plotnine.scale_linetype_manual(values={"Train": "solid", "Validation": "dashed"})
     + plotnine.labs(color="Trial", linetype="Loss Type")
)

# Save the plot to a file
if not os.path.exists("Analyses_outputs/Plots/Fine_Tuning"):
    os.makedirs("Analyses_outputs/Plots/Fine_Tuning")

p.save("Analyses_outputs/Plots/Fine_Tuning/training_validation_loss_comparison_OpenAI.png", width=19.2, height=9.67, dpi=300)

p.draw()

## Optional: Estimating inference cost of the fine-tuned model for whole dataset

Last, we will estimate the cost of using the fine-tuned model for the classification task.

In [ ]:
def count_message_tokens(messages, model="gpt-4.1-nano-2025-04-14"):
    encoding = tiktoken.encoding_for_model(model)
    tokens_per_message = 3  # metadata overhead per message (approx.)
    tokens_per_name = 1     # additional if "name" field present

    total_tokens = 0
    for msg in messages:
        total_tokens += tokens_per_message
        for key, value in msg.items():
            total_tokens += len(encoding.encode(value))
            if key == "name":
                total_tokens += tokens_per_name
    total_tokens += 3  # reply priming
    return total_tokens

In [ ]:
# Load the final cleaned dataframe for batch processing
df_final = pickle.load(open("GLLM/Corpus_df_HTML_cleaned_GLLM_final.pkl", "rb"))

# create new column for sentence_id
df_final['sentence_ids'] = None

prompt_template = """Text:
{text}
####
"""

# Create a jsonl file for the whole dataset
file_name = "GLLM/OpenAI_batch_files/MAP_sentences_final_FT.jsonl"

tasks = []

model = "gpt-4.1-mini-2025-04-14"

processing_file = open(file_name, "w", encoding="utf-8")

for index, row in df_final.iterrows():
    filing_id = f"filing_{index}"

    df_final.at[index, 'filing_id'] = filing_id

    sentence_ids = []

    total_tokens = 0
    for sentence in row['filing_text']:
        
        if not sentence or pd.isna(sentence):  # Skip empty or NaN sentences
            continue

        sentence_id = f"filing_{index}_sentence_{row['filing_text'].index(sentence)}"
        sentence_ids.append(sentence_id)
        task = {
            "custom_id": f"{sentence_id}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model,
                "temperature": 0.001,
                "response_format": {
                    "type": "json_object"
                },
                "messages": [
                    {
                        "role": "user",
                        "content": prompt_template.format(text=sentence)
                    }
                ],
            }
        }
        processing_file.write(json.dumps(task) + "\n")
        processing_file.flush()

        total_tokens += count_message_tokens(task['body']['messages'], model="gpt-4.1-mini-2025-04-14")


    df_final.at[index, 'sentence_ids'] = sentence_ids
    df_final.at[index, 'total_tokens'] = total_tokens

processing_file.close()

In [ ]:
#total input tokens for the whole dataset
total_input_tokens = df_final["total_tokens"].sum()
print(f"{total_input_tokens} total input tokens for the whole dataset. This would take around {total_input_tokens/40_000_000:.2f} days to process at 40M tokens per day (Limit for Tier 3).")

# Cost estimate at 0.2$ per 1M input tokens
cost_estimate = total_input_tokens / 1_000_000 * 0.2
print(f"Estimated cost at $0.20 per 1M input tokens: ${cost_estimate:.2f}")

#total output tokens at an estimate of 41 tokens per completion
total_prompts = df_final["num_entries"].sum()
total_output_tokens = total_prompts * 41
print(f"{total_output_tokens} total output tokens for the whole dataset")

# Cost estimate at 0.80$ per 1M output tokens
cost_estimate_output = total_output_tokens / 1_000_000 * 0.80
print(f"Estimated cost at $0.80 per 1M output tokens: ${cost_estimate_output:.2f}")

# Total estimated cost
total_estimated_cost = cost_estimate + cost_estimate_output
print(f"Total estimated cost: ${total_estimated_cost:.2f}")